In [21]:
import numpy as np
import pandas as pd

class CART:
    def __init__(self, max_depth=5): ### max_depth = 5 espace de stockage 
        self.tree = None  ### eviter la sur-apprentissage 
        self.max_depth = max_depth ### enregistremet de réglage

    #### calculation de gini cette methode de calcule est rapide par rapport de l'entropie ( précise )
    def gini_impurity(self, y):
        """ Calcule l'impureté de Gini au lieu de l'Entropy """
        if len(y) == 0: return 0
        probs = [np.mean(y == c) for c in np.unique(y)] ## Pi probabilité de chaque attribut dans l'échantillon
        # Formule  de gini :gini (s) = 1 - sum(pi^2)
        return 1 - np.sum([p**2 for p in probs])
   #### la séparation de dataset
    def split_data(self, X, y, feature_idx, value):
        """ Division binaire : split en deux groupes (Gauche et Droite) """
        left_mask = X[:, feature_idx] == value
        right_mask = ~left_mask
        return X[left_mask], y[left_mask], X[right_mask], y[right_mask]

    def best_split(self, X, y, features):
        """ Trouve la meilleure division qui réduit le Gini le plus possible """
        best_gini = float('inf')
        split_info = None

        for i in range(len(features)):
            values = np.unique(X[:, i])
            for val in values:
                # On teste chaque valeur comme point de division
                X_l, y_l, X_r, y_r = self.split_data(X, y, i, val)
                
                if len(y_l) == 0 or len(y_r) == 0: continue

                # Gini pondéré des deux nouveaux groupes
                p_l = len(y_l) / len(y)
                p_r = len(y_r) / len(y)
                current_gini = p_l * self.gini_impurity(y_l) + p_r * self.gini_impurity(y_r)

                if current_gini < best_gini:
                    best_gini = current_gini
                    split_info = {'feature_idx': i, 'feature_name': features[i], 'value': val, 
                                 'left': (X_l, y_l), 'right': (X_r, y_r)}
        return split_info

    def fit(self, X, y, features):
        self.tree = self._build_tree(X, y, features)

    def _build_tree(self, X, y, features, depth=0):
        # Conditions d'arrêt (Max depth ou pureté)
        if len(np.unique(y)) == 1 or depth >= self.max_depth or len(y) < 2:
            return np.unique(y)[0]

        # Trouver le meilleur split binaire
        split = self.best_split(X, y, features)
        if not split:
            return np.unique(y)[np.argmax(np.unique(y, return_counts=True)[1])]

        # Construction récursive de l'arbre binaire
        node = {
            'question': f"{split['feature_name']} == {split['value']}",
            'left': self._build_tree(*split['left'], features, depth + 1),
            'right': self._build_tree(*split['right'], features, depth + 1)
        }
        return node

    def predict(self, sample, tree=None):
        if tree is None: tree = self.tree
        if not isinstance(tree, dict): return tree

        # Logique binaire : Si oui -> gauche, si non -> droite
        feature_name = tree['question'].split(' == ')[0]
        value = tree['question'].split(' == ')[1]

        if str(sample[feature_name]) == value:
            return self.predict(sample, tree['left'])
        else:
            return self.predict(sample, tree['right'])
            

# --- TEST AVEC LE DATASET TENNIS ---
data = {
   'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 
                'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temp': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 
             'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 
                 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 
             'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'Play': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 
             'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}

df = pd.DataFrame(data)
X = df.drop('Play', axis=1).values
y = df['Play'].values
features = ['Outlook', 'Temp','Humidity','Wind']

model = CART()
model.fit(X, y, features)

import pprint
print("--- Arbre CART (Binaire avec Gini) ---")
pprint.pprint(model.tree)

--- Arbre CART (Binaire avec Gini) ---
{'left': 'Yes',
 'question': 'Outlook == Overcast',
 'right': {'left': {'left': {'left': 'No',
                             'question': 'Wind == Strong',
                             'right': 'Yes'},
                    'question': 'Outlook == Rain',
                    'right': 'No'},
           'question': 'Humidity == High',
           'right': {'left': {'left': 'No',
                              'question': 'Outlook == Rain',
                              'right': 'Yes'},
                     'question': 'Wind == Strong',
                     'right': 'Yes'}}}
